In [ ]:
import tempfile
import httpx
import rich.progress

In [ ]:
with tempfile.NamedTemporaryFile() as download_file:
    url = "https://raw.githubusercontent.com/gitinference/jp-tools/refs/heads/main/LICENSE"
    with httpx.stream("GET", url) as response:
        total = int(response.headers["Content-Length"])

        with rich.progress.Progress(
            "[progress.percentage]{task.percentage:>3.0f}%",
            rich.progress.BarColumn(bar_width=None),
            rich.progress.DownloadColumn(),
            rich.progress.TransferSpeedColumn(),
        ) as progress:
            download_task = progress.add_task("Download", total=total)
            for chunk in response.iter_bytes():
                download_file.write(chunk)
                progress.update(download_task, completed=response.num_bytes_downloaded)

In [ ]:
import tempfile

import httpx
from tqdm import tqdm

with tempfile.NamedTemporaryFile() as download_file:
    url = "https://speed.hetzner.de/100MB.bin"
    with httpx.stream("GET", url) as response:
        total = int(response.headers["Content-Length"])

        with tqdm(total=total, unit_scale=True, unit_divisor=1024, unit="B") as progress:
            num_bytes_downloaded = response.num_bytes_downloaded
            for chunk in response.iter_bytes():
                download_file.write(chunk)
                progress.update(response.num_bytes_downloaded - num_bytes_downloaded)
                num_bytes_downloaded = response.num_bytes_downloaded


In [ ]:
def download(
    url: str, filename: str, verify: bool = True, notify_flag: bool = False
) -> None:
    """
    Pulls a file from a URL and saves it in the filename. Used by the class to pull external files.

    Parameters
    ----------
    url: str
        The URL to pull the file from.
    filename: str
        The filename to save the file to.
    verify: bool
        If True, verifies the SSL certificate. If False, does not verify the SSL certificate.

    Returns
    -------
    None
    """
    if os.path.exists(filename):
        return None

    chunk_size = 10 * 1024 * 1024

    with requests.get(url, stream=True, verify=verify) as response:
        total_size = int(response.headers.get("content-length", 0))

        with tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Downloading",
        ) as bar:
            with open(filename, "wb") as file:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        file.write(chunk)
                        bar.update(
                            len(chunk)
                        )  # Update the progress bar with the size of the chunk
    if notify_flag:
        notify(
            url=str(os.environ.get("URL")),
            auth=str(os.environ.get("AUTH")),
            msg=f"Successfully downloaded {filename} from {url}",
        )

download(
        url="https://raw.githubusercontent.com/gitinference/jp-tools/refs/heads/main/LICENSE",
        filename="DELETE.txt",
    )